# 1. Environment Setup & Dependency Imports

In this section, we initialize the required environment and import core modules for document extraction, tabular parsing, dense vector retrieval, and local LLM execution.

* **Document Parsing**: `PyMuPDF` (`fitz`) for PDFs and `pandas` for tabular data.
* **Text Chunking**: `RecursiveCharacterTextSplitter` with overlapping windows.
* **Dense Retrieval**: `FAISS` vector store integrated with local `OllamaEmbeddings`.
* **LLM Orchestration**: `ChatOllama` running `llama3.2` locally via LangChain Expression Language (LCEL).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pymupdf  # Fast PDF text extraction library
import pandas as pd  # Data manipulation library for tabular files (Excel)

# LangChain modules for document management and text splitting
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# FAISS vector store for similarity search and dense retrieval
from langchain_community.vectorstores import FAISS

# Prompt templates and LCEL (LangChain Expression Language) components
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import AIMessage, HumanMessage

# Local Ollama integration for Embeddings and LLM execution
from langchain_ollama import ChatOllama, OllamaEmbeddings

# 2. Multi-Format Institutional Data Parsing Pipeline
    Extracts text and tabular data from institutional reports (Bank Al-Maghrib, HCP, DEPF, Academic Papers).
* **Unstructured Processing (PDF)**: Iterates page-by-page to extract raw text while attaching page-level metadata.\n",
* **Structured Processing (Excel)**: Converts dataframes into Markdown table structures to preserve tabular semantics."

In [ ]:
def parse_all_macro_data(base_dir):

    all_docs = []
    base_path = Path(base_dir)

    # Processing Unstructured PDF Files 
    for folder in ["BAM", "HCP", "PAPERS"]:
        folder_path = base_path / folder
        if not folder_path.exists():
            continue
            
        # Iterate over each PDF in the directory
        for pdf_file in folder_path.glob("*.pdf"):
            try:
                pdf_doc = pymupdf.open(pdf_file)
                # Page-by-page iteration to track precise page metadata
                for page_num in range(len(pdf_doc)):
                    text = pdf_doc[page_num].get_text("text").strip()
                    if text:
                        # Append each page as a Document with associated metadata
                        all_docs.append(Document(
                            page_content=text,
                            metadata={
                                "source": str(pdf_file),  # Standard retriever metadata key
                                "file_path": str(pdf_file),
                                "file_name": pdf_file.name,
                                "folder": folder,
                                "page": page_num + 1,
                                "file_type": "pdf"
                            }
                        ))
                pdf_doc.close()
            except Exception as e:
                print(f"Error parsing PDF {pdf_file.name}: {e}")

    #  Processing Structured Excel Files 
    depf_path = base_path / "DEPF"
    if depf_path.exists():
        for excel_file in depf_path.glob("*.xlsx"):
            try:
                # Read all sheets in the Excel workbook
                sheets = pd.read_excel(excel_file, sheet_name=None)
                for sheet_name, df in sheets.items():
                    # Clean completely empty rows and columns
                    df_clean = df.dropna(how='all').dropna(how='all', axis=1)
                    if not df_clean.empty:
                        # Convert dataframe to Markdown format to preserve table semantics for the LLM
                        markdown_table = df_clean.to_markdown(index=False)
                        content = f"# DEPF - {excel_file.name}\n## Sheet: {sheet_name}\n\n{markdown_table}"
                        all_docs.append(Document(
                            page_content=content,
                            metadata={
                                "source": str(excel_file),
                                "file_path": str(excel_file),
                                "file_name": excel_file.name,
                                "folder": "DEPF",
                                "sheet_name": sheet_name,
                                "file_type": "excel"
                            }
                        ))
            except Exception as e:
                print(f"Error parsing Excel {excel_file.name}: {e}")

    return all_docs

# Dynamically resolve project root directory
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    PROJECT_ROOT = BASE_DIR.parent
else:
    PROJECT_ROOT = BASE_DIR

DATA_DIR = PROJECT_ROOT / "data"

# Execute document parsing pipeline
documents = parse_all_macro_data(DATA_DIR)
print(f"Total extracted documents/pages: {len(documents)}")

# 3. PDF Extraction Verification

Sanity check to inspect raw text extractions from target PDFs and ensure character encoding and formatting integrity.

In [ ]:
# Manual verification check on a specific target PDF
sample_pdf = PROJECT_ROOT / "data" / "BAM" / "BAM-RA20-Fr.pdf"
if sample_pdf.exists():
    pdf_doc = pymupdf.open(sample_pdf)
    # Extract raw text from page 25 (zero-indexed at 24)
    page_text = pdf_doc[24].get_text("text")

    print("--- RAW TEXT EXTRACTION BY PYMUPDF ---")
    print(page_text[:1000])
else:
    print(f"Sample PDF not found at path: {sample_pdf}")

# 4. Text Chunking

Splits full-length documents into smaller chunks to maintain precise context windows for embedding generation.

In [ ]:
def chunk_documents(documents):
    """
    Splits full-length documents into smaller chunks with overlap to preserve context semantics.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,        # Maximum chunk length in characters
        chunk_overlap=150,     # Overlap length between adjacent chunks
        separators=["\n\n", "\n", ".", " "]  # Separator hierarchy order
    )
    return splitter.split_documents(documents)

# Execute text chunking on extracted documents
chunked_docs = chunk_documents(documents)
print(f"Total chunks created: {len(chunked_docs)}")

# 5. Dense Embedding & FAISS Vector Store Indexing
 Text chunks are mapped to high-dimensional dense vector spaces using `nomic-embed-text` via Ollama.
 The generated vectors are stored in a local **FAISS** (Facebook AI Similarity Search) index.

In [ ]:
def create_vector_store(chunked_docs):
    """
    Generates dense embeddings using nomic-embed-text via Ollama
    and initializes the local FAISS vector store.
    """
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vector_store = FAISS.from_documents(chunked_docs, embedding=embeddings)
    return vector_store

# Create local FAISS vector store index
vector_store = create_vector_store(chunked_docs)

# Inspect vector dimensions and layout for the first 5 chunks
vectors = vector_store.index.reconstruct_n(0, 5)
print(f"Vector dimensions (count, dimension): {np.array(vectors).shape}")
print(f"First 10 values of the 1st vector: {vectors[0][:10]}")

# 6. Similarity Search with Threshold Filtering
Retrieves top-k documents and applies a minimum relevancy score cut-off to prevent hallucinations

In [ ]:
def generate_answer(vector_store, query, k=5, score_threshold=0.3):
    """
    Retrieves top-k similar documents and filters results based on a minimum relevance threshold.
    """
    # Retrieve top-k documents along with relevance scores
    docs = vector_store.similarity_search_with_relevance_scores(query, k=k)
    
    # Filter documents based on score cutoff to reduce hallucinations
    filtered_docs = [doc for doc, score in docs if score >= score_threshold]
    if not filtered_docs:
        return "No useful content"
    
    # Format retrieved context blocks with page-level metadata
    context_blocks = []
    for doc in filtered_docs:
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', "N/A")
        context_blocks.append(f"[Source:{source} | Page:{page}]\n{doc.page_content}")
        
    return "\n\n---\n\n".join(context_blocks)

# 7. Stateless RAG Pipeline Execution
Construction of a single-turn LCEL (LangChain Expression Language) runnable chain.
**Strict Grounding Prompt**: Instructs the LLM (`llama3.2`) to answer strictly from context and explicitly declare missing information.
Uses token streaming for real-time output rendering.

In [ ]:
# Configure vector retriever from FAISS vector store
retriever = vector_store.as_retriever(
    search_type="similarity", search_kwargs={"k": 5}
)

# Strict system prompt preventing extrapolation beyond provided context
SYSTEM_PROMPT = """Answer the user's question based ONLY on the provided context below.
If the answer cannot be found in the context, state clearly: "I cannot find this information in the provided document."
Do not attempt to invent, extrapolate, or use outside knowledge.
Context : {context}
Question : {question}
Answer : """

prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)
llm = ChatOllama(model="llama3.2", temperature=0)  # Temperature=0 for deterministic outputs
output_parser = StrOutputParser()

def format_docs(docs):
    """Formats retrieved Document list into a single structured context string."""
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('source', 'N/A')} | Page: {d.metadata.get('page', 'N/A')}]\n{d.page_content}"
        for d in docs
    )

# Assemble single-turn LCEL runnable chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

# Stream generated response tokens in real-time
for chunk in rag_chain.stream("Quel est le résultat net de Bank Al-Maghrib en 2020 ?"):
    print(chunk, end="", flush=True)
print()

# 8. Conversational RAG with Contextual Query Rewriting
To handle multi-turn dialogue with coreference resolution (e.g., *'Et quel était son montant en 2019 ?'*):
1. **Query Rewriter**: Reformulates follow-up queries into standalone search inputs using past conversation history.
2. **Contextual Generation**: Executes retrieval with the standalone query and answers the original prompt in conversation context.

In [ ]:
#  Query Rewriter Prompt for Coreference Resolution 
REWRITE_SYSTEM_PROMPT = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", REWRITE_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# Main Question-Answering Prompt 
QA_SYSTEM_PROMPT = """Answer the user's question based ONLY on the provided context below.
If the answer cannot be found in the context, state clearly: "I cannot find this information in the provided document."
Do not attempt to invent, extrapolate, or use outside knowledge.
Context : \n{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", QA_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

llm = ChatOllama(model="llama3.2", temperature=0)
output_parser = StrOutputParser()
question_rewriter = rewrite_prompt | llm | output_parser

def format_docs(docs):
    """Formats retrieved document list with fallback handling for empty results."""
    if not docs:
        return "No context found"
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('source', 'Unknown')} | Page: {d.metadata.get('page', 'N/A')}]\n{d.page_content}"
        for d in docs
    )

def conversational_rag(user_input, chat_history, retriever):
    """
    Handles multi-turn conversational RAG workflow:
    1. Reformulates the query into a standalone question if dialogue history exists.
    2. Queries FAISS retriever using the standalone question.
    3. Streams response generation using Llama 3.2.
    """
    # Step 1: Query reformulation if chat history is present
    if chat_history:
        standalone_question = question_rewriter.invoke({"chat_history": chat_history, "input": user_input})
    else:
        standalone_question = user_input

    # Step 2: Dense retrieval using reformulated question
    docs = retriever.invoke(standalone_question)
    context = format_docs(docs)
    
    # Step 3: Streamed response generation
    rag_chain = qa_prompt | llm | output_parser
    final_response = ""
    for chunk in rag_chain.stream({"context": context, "chat_history": chat_history, "input": user_input}):
        print(chunk, end="", flush=True)
        final_response += chunk
    
    print()
    return final_response

# 9. Main Pipeline Execution & Multi-Turn Verification 
    Entry point verifying end-to-end multi-turn conversation over local data directory structure.

In [ ]:
if __name__ == "__main__":
    # Environment-configurable path for target documents directory
    DATA_DIR = os.getenv("DATA_DIR", "./data")

    if not os.path.exists(DATA_DIR):
        print(f"Data directory '{DATA_DIR}' not found. Please create it and add source files.")
    else:
        print("--- Parsing Data Documents ---")
        raw_documents = parse_all_macro_data(DATA_DIR)
        print(f"Total extracted documents/pages: {len(raw_documents)}")

        print("\n--- Chunking Documents ---")
        chunks = chunk_documents(raw_documents)
        print(f"Total chunks created: {len(chunks)}")

        print("\n--- Generating Embeddings & FAISS Vector Store ---")
        vector_store = create_vector_store(chunks)
        retriever = vector_store.as_retriever(
            search_type="similarity", search_kwargs={"k": 5}
        )

        print("\n--- Testing Conversational RAG Pipeline ---")
        chat_history = []

        #  First Conversation Turn 
        query_1 = "Quel est le résultat net de Bank Al-Maghrib en 2020 ?"
        print(f"\nUser: {query_1}")
        print("Assistant: ", end="")

        response_1 = conversational_rag(query_1, chat_history, retriever)
        chat_history.append(HumanMessage(content=query_1))
        chat_history.append(AIMessage(content=response_1))

        #  Second Conversation Turn (with coreference resolution) 
        query_2 = "Et quel était son montant en 2019 ?"
        print(f"\nUser: {query_2}")
        print("Assistant: ", end="")

        response_2 = conversational_rag(query_2, chat_history, retriever)